In [ ]:
# Testing Cell
import inspect

import aviary.api as av
from aviary.subsystems.aerodynamics.aerodynamics_builder import CoreAerodynamicsBuilder
from aviary.utils.doctape import get_variable_name, glue_variable

current_glued_vars = []

CoreAerodynamicsBuilder
glue_variable(get_variable_name(CoreAerodynamicsBuilder), md_code=True)

HEIGHT_ENERGY = av.EquationsOfMotion.HEIGHT_ENERGY
glue_variable('height_energy', HEIGHT_ENERGY.value, md_code=False)

TWO_DEGREES_OF_FREEDOM = av.EquationsOfMotion.TWO_DEGREES_OF_FREEDOM
glue_variable(TWO_DEGREES_OF_FREEDOM.value, md_code=False)

# glue all arguments of function CoreAerodynamicsBuilder.__init__()
sigs = inspect.signature(CoreAerodynamicsBuilder)
parameters = sigs.parameters
for name, param in parameters.items():
    glue_variable(name, md_code=True)
    # print(f'Name: {name}, Default: {param.default}, Kind: {param.kind}')

<!-- Work in progress... -->
# RC-Aircraft/Electric UAV Modeling

Some background on dbf and why its important
<!-- The built-in aerodynamics subsystem in Aviary offers multiple options for computing drag. Users can select from methods based on the FLOPS or GASP legacy codes. Choice of which legacy code's routines to use is determined by the {glue:md}`code_origin` option provided when initializing a {glue:md}`CoreAerodynamicsBuilder`. When using Aviary's [Level 1 interface](../getting_started/onboarding_level1), the code origin for aerodynamics is automatically set to match with the mission method ({glue:md}`height_energy` is paired with FLOPS, and {glue:md}`2DOF` is paired with GASP). Future updates to Aviary will allow for the user to specify aerodynamics code origin directly in the input file. -->

## Propulsion
The electric propulsion system for an RC-aircraft typically involves a battery, blade fuse, an electronic speed controller, a motor, and the propeller that provides the thrust for the aircraft. In each of these components there are losses from the power provided by the battery to the motor, all of which are significant enough to consider with the exception of the blade fuse. In this section, we'll take a look at each of these components, and how they are linked together in a way that allows for optimization of your propeller-airframe integration with either provided performance requirements or in the context of a pre-described mission. 
<!-- The [`rc_performance.py`](https://github.com/OpenMDAO/Aviary/blob/main/aviary/subsystems/propulsion/rc_electric/model/rc_performance.py) file contains the source code for this; which we recommend you look at for a more detailed explanation. -->
### Battery
As of now, the battery model directly integrated into the RC-aircraft model is incredibly simplistic, and uses Ohm's law to compute the losses from the battery's resistance and the electric current flowing from said battery. The voltage of the battery in this case is a user-provided value, and is not optimized throughout the optimization. To compute the total energy capacity from a battery, we use a linear relationship between the battery mass and the nominal capacity provided by a particular manufacturer, and multiply the nominal capacity by the initial voltage. Because of this tradeoff between the mass of the battery and its energy capacity, we are able to optimize the mass of the energy for different flight profiles, which is included as a Pre-Mission system. 
The shortcomings of this method are:
1. The voltage of the battery is constant throughout the mission, thus not considering losses for longer flights.
2. The linear relationship between battery mass and nominal capacity varies depending on the number of cells in a battery. The current implementation is from 6 cell batteries, as they are often found in the Design Build Fly competitions this model was originally intended for. 
3. This battery model is curently inputted directly into the ['EngineModel'] group. As a result, there is no way for a user to declare they have two motors but only one battery providing voltage to both. 

**In the future, the solution to these problems is to use the Thevenin Circuit Model from the battery external subsystem already in Aviary, as this would increase the flexibility of the component, and allows for the state of charge to be added as a state in a trajectory optimization.**

### Electronic Speed Controller
Work in progress

### Motor
Work in progress

### Propeller 
To properly simulate the performance of small scale propellers would be difficult at each iteration given the geometry as design variables. To resolve this, we use surrogate modeling techniques and [APC Propeller](https://www.apcprop.com/technical-information/performance-data/?v=7516fd43adaa) performance data. 
For a given propeller geometry, i.e 10x5, (where 10 denotes the diameter in inches and 5 is the pitch in inches), data varies in airspeed and RPM; providing thrust and power coefficients $C_T$, $C_P$, and torque $Q$ as outputs. We verify this using the [UIUC Propeller Site](https://m-selig.ae.illinois.edu/props/propDB.html) experimental wind tunnel data. 
![Advance ratio v performance coefficients for wind tunnel and simulation data](images/APCCheck.png)
While APC has over thousands of individual propellers simulated, we use only thin electric data, as these propellers are typical of the aircrafts we are attempting to model. The performance coefficients for each propeller can be simply characterized based on the advance ratio, $J=\frac{V}{\omega D}$, where $D$ is the diameter, $V$ is the aircraft's forward velocity, and $\omega$ is the rotational speed (RPS) of the propeller. Unfortunately, to take into consideration static conditions, characterizing each propeller based solely on their pitch and advance ratio is not possible, as for a same pitched propeller with different diameters the advanced ratio for both would be 0. Thus, we model the thrust and power coefficients as $C_T, C_P = f(D, \rho, RPM, V)$, where $\rho$ is the pitch of the propeller. To allow for input values that are not found in the data, we implement a [Semi-Structured Meta Model](https://openmdao.org/newdocs/versions/latest/features/building_blocks/components/metamodelsemistructured_comp.html), and use a second order lagrange polynomial to interpolate through the inputs. The full range of parsed data can be found in the folder ['Parsing'](https://github.com/OpenMDAO/Aviary/blob/main/aviary/subsystems/propulsion/rc_electric/Parsing), in case you choose to add to this data or modify it for a certain propeller distributor. While the full data ranges from diameters of 4-27 inches, the current MetaModel uses propellers between 12-20 inch diameters, as these are typical in DBF aircrafts, and not enough interpolation points are provided by the data to construct the model at larger dimensions. 
Using $C_T$ and $C_P$, we then compute the power and thrust outputted by the propeller. 
### Linking
To link these components, we implicitly solve for the current using a residual of the total power in the system:

$$R(I) = P_{battery} - P_{esc} - P_{motor} + P_{prop}$$

We subtract the power from the esc and the motor because these are the power losses, whereas the battery and propeller power is that outputted by the component. From the perspective of performance of a trajectory optimization, it was found more efficient to give this residual as a constraint for the optimizer, meaning that if the optimization were to end early, the values in the model may not be physically accurate. The entire subsystem is represented via the following XDSM:

**ADD**

These computations are performed at each point in the mission, and must be done twice in cases in which the maximum thrust possible must be computed (for certain ODEs), which is represented by solving for the current implicitly at the same flight condition but at full throttle. 

